# CommGuard benign corpus v2

Restore a hash-pinned supported calibration artifact, run a bounded pilot by default, and preserve coverage before any detector evaluation. The standard 24-run corpus is an explicit opt-in.


In [ ]:
from pathlib import Path
import re
import subprocess
import sys

NOTEBOOK_VERSION = "commguard_benign_corpus_v2"
REPOSITORY_URL = "https://github.com/waqasm86/CommGuard.git"
REVIEWED_COMMIT = ""  # Required: immutable 40-character commit visible on origin.
REPOSITORY = Path("/kaggle/working/commguard-source")

if not re.fullmatch(r"[0-9a-f]{40}", REVIEWED_COMMIT):
    raise RuntimeError("Set REVIEWED_COMMIT to the reviewed, pushed 40-character commit SHA.")
if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, str(REPOSITORY)],
        check=True,
    )
if not (REPOSITORY / ".git").is_dir():
    raise RuntimeError(f"Refusing non-Git source directory: {REPOSITORY}")
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "origin", REVIEWED_COMMIT], check=True)
subprocess.run(
    ["git", "-C", str(REPOSITORY), "checkout", "--detach", REVIEWED_COMMIT], check=True
)
head = subprocess.run(
    ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(REPOSITORY), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
pushed_refs = subprocess.run(
    ["git", "-C", str(REPOSITORY), "branch", "-r", "--contains", head],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if head != REVIEWED_COMMIT or dirty or not pushed_refs:
    raise RuntimeError(
        "Reproducibility gate failed: "
        f"head={head} dirty={bool(dirty)} pushed={bool(pushed_refs)}"
    )
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps",
        "-e", str(REPOSITORY),
    ],
    check=True,
)
print({"reviewed_commit": head, "remote_refs": pushed_refs.splitlines()})


In [ ]:
from commguard.artifacts import restore_archive, sha256_file

INPUT_ARCHIVE = Path("/kaggle/input/commguard-calibration-v3/commguard-calibration-v3-REPLACE.tar.gz")
EXPECTED_INPUT_SHA256 = ""  # Required: SHA-256 printed by the preceding notebook.
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")

if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_INPUT_SHA256):
    raise RuntimeError("Set EXPECTED_INPUT_SHA256 to the exact 64-character archive hash.")
actual_input_sha256 = sha256_file(INPUT_ARCHIVE)
if actual_input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError(
        "Input archive hash mismatch: "
        f"expected={EXPECTED_INPUT_SHA256} actual={actual_input_sha256}"
    )
restore_archive(INPUT_ARCHIVE, ARTIFACTS, expected_sha256=EXPECTED_INPUT_SHA256)
print({"restored_archive": str(INPUT_ARCHIVE), "sha256": actual_input_sha256})


In [ ]:
from datetime import datetime, timezone

from commguard.environment.preflight import check_environment, summarize_environment
from commguard.provenance import ProvenanceContext

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
CONTEXT = ProvenanceContext.create(
    corpus_id=f"corpus-benign-v2-{NOTEBOOK_RUN_ID}",
    experiment_session_id=f"session-benign-v2-{NOTEBOOK_RUN_ID}",
    collection_id=f"collection-benign-v2-{NOTEBOOK_RUN_ID}",
    notebook_version="commguard_benign_corpus_v2",
    input_archive_sha256=EXPECTED_INPUT_SHA256,
    random_seed=20260730,
    repository_root=REPOSITORY,
)
if CONTEXT.source_dirty or CONTEXT.source_commit != REVIEWED_COMMIT:
    raise RuntimeError("SDK provenance no longer matches the clean reviewed source commit.")
ENVIRONMENT = check_environment(strict=True, output=ARTIFACTS, provenance=CONTEXT)
print(summarize_environment(ENVIRONMENT))
print({
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "experiment_session_id": CONTEXT.experiment_session_id,
    "collection_id": CONTEXT.collection_id,
    "corpus_id": CONTEXT.corpus_id,
    "source_commit": CONTEXT.source_commit,
    "input_archive_sha256": CONTEXT.input_archive_sha256,
})


In [ ]:
import json

calibration_paths = sorted((ARTIFACTS / "results").glob("calibration-*.json"))
if not calibration_paths:
    raise RuntimeError("The restored archive has no calibration result.")
INPUT_CALIBRATION = json.loads(calibration_paths[-1].read_text(encoding="utf-8"))
if INPUT_CALIBRATION.get("status") != "supported":
    status = INPUT_CALIBRATION.get("status")
    raise RuntimeError(f"Benign collection blocked by calibration={{status}}")
print({"accepted_calibration": str(calibration_paths[-1]), "status": "supported"})


In [ ]:
from commguard.orchestrator import estimate_matrix, run_matrix

RUN_BENIGN_PILOT = True
RUN_STANDARD_BENIGN_MATRIX = False
RUN_EXPANDED_BENIGN_MATRIX = False

requested = []
if RUN_BENIGN_PILOT:
    requested.append(("smoke", 1))
if RUN_STANDARD_BENIGN_MATRIX:
    requested.append(("standard", 3))
if RUN_EXPANDED_BENIGN_MATRIX:
    requested.append(("extended", 3))
if len(requested) != 1:
    raise RuntimeError("Enable exactly one benign profile per immutable notebook archive.")
PROFILE, REPETITIONS = requested[0]
print({"estimate": estimate_matrix(PROFILE, REPETITIONS), "duration_aware": True})
MATRIX = run_matrix(
    PROFILE,
    output=ARTIFACTS,
    repetitions=REPETITIONS,
    timeout_s=180.0,
    provenance=CONTEXT,
)


In [ ]:
COVERAGE_TABLE = [
    {"family": family, **counts}
    for family, counts in sorted(MATRIX["family_counts"].items())
]
for row in COVERAGE_TABLE:
    print(row)
print({
    "primary_coverage_gate": MATRIX["primary_coverage_gate"],
    "detector_metrics_computed": MATRIX["detector_metrics_computed"],
})


## Results

not executed. Coverage and completion are unknown until the notebook is run.


In [ ]:
from commguard.artifacts import ArtifactStore, sha256_file

ARCHIVE = Path(f"/kaggle/working/commguard-benign-corpus-v2-{NOTEBOOK_RUN_ID}.tar.gz")
ArtifactStore(ARTIFACTS).export(ARCHIVE)
ARCHIVE_SHA256 = sha256_file(ARCHIVE)
SHA_FILE = ARCHIVE.with_suffix(ARCHIVE.suffix + ".sha256")
SHA_FILE.write_text(f"{ARCHIVE_SHA256}  {ARCHIVE.name}\n", encoding="utf-8")
print(f"NEXT STEP: add {ARCHIVE} to a private Kaggle dataset without renaming it.")
print(f"NEXT STEP: copy SHA-256 {ARCHIVE_SHA256} into EXPECTED_INPUT_SHA256 in commguard_detector_evaluation_v2.ipynb.")
print(
    f"NEXT STEP: set that notebook's REVIEWED_COMMIT to {REVIEWED_COMMIT} "
    "and run from the first cell."
)
